In [ ]:
import os
import time
import json
import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, accuracy_score
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

tf.random.set_seed(42)

IMG_ROOT = '../data/Images'
IMG_SIZE = (128, 128)
label_map = {
    'Negative': 'negative',
    'Neutral':  'neutral',
    'positive': 'positive',
}

## Load all images into memory

MobileNetV2 expects pixels in [-1, 1] via `preprocess_input` rather than [0, 1].

In [ ]:
def load_image(path):
    img = Image.open(path).convert('RGB')
    img = img.resize(IMG_SIZE, Image.LANCZOS)
    arr = np.array(img, dtype='float32')
    return preprocess_input(arr)  # scales to [-1, 1]

records = []
for folder, label in label_map.items():
    folder_path = os.path.join(IMG_ROOT, folder)
    for fname in os.listdir(folder_path):
        if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
            records.append({'path': os.path.join(folder_path, fname), 'label': label})

df = pd.DataFrame(records)

print(f'Loading {len(df)} images...')
X = np.array([load_image(p) for p in df['path']])
print(f'Loaded. Array shape: {X.shape}  Memory: {X.nbytes / 1e9:.2f} GB')

le = LabelEncoder()
y = le.fit_transform(df['label'])
print('Classes:', le.classes_)

## Train/val/test split

In [ ]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=0.2,
    random_state=42,
    stratify=y_trainval
)

print(f'Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}')

## Tune classification head on validation set

MobileNetV2 base is frozen (weights='imagenet'). We tune the number of units in the Dense head.

In [ ]:
def build_model(dense_units):
    base = MobileNetV2(input_shape=(128, 128, 3), include_top=False, weights='imagenet')
    base.trainable = False
    model = Sequential([
        base,
        GlobalAveragePooling2D(),
        Dense(dense_units, activation='relu'),
        Dropout(0.5),
        Dense(3, activation='softmax')
    ])
    return model

t0 = time.time()
units_options = [128, 256]
val_results = []
es = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

for units in units_options:
    model = build_model(units)
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=20,
        batch_size=32,
        callbacks=[es],
        verbose=0
    )
    val_acc = model.evaluate(X_val, y_val, verbose=0)[1]
    val_results.append({'dense_units': units, 'val_acc': val_acc})
    print(f'dense_units={units:<4}  Val Accuracy: {val_acc:.3f}')

best_units = max(val_results, key=lambda x: x['val_acc'])['dense_units']
print(f'\nBest dense_units: {best_units}')

## Retrain on full train+val, evaluate on test

In [ ]:
best_model = build_model(best_units)
best_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
best_model.fit(
    X_trainval, y_trainval,
    epochs=100,
    batch_size=32,
    callbacks=[EarlyStopping(monitor='loss', patience=10, restore_best_weights=True)],
    verbose=1
)
runtime = time.time() - t0

y_pred_enc = np.argmax(best_model.predict(X_test), axis=1)
y_pred = le.inverse_transform(y_pred_enc)
y_test_labels = le.inverse_transform(y_test)

print(f'\nAccuracy: {accuracy_score(y_test_labels, y_pred):.3f}')
print(f'Runtime:  {runtime:.2f}s')
print()
print(classification_report(y_test_labels, y_pred))

report = classification_report(y_test_labels, y_pred, output_dict=True)
meta = {
    'model': 'cnn_mobilenet',
    'accuracy': report['accuracy'],
    'macro_f1': report['macro avg']['f1-score'],
    'negative_f1': report['negative']['f1-score'],
    'neutral_f1': report['neutral']['f1-score'],
    'positive_f1': report['positive']['f1-score'],
    'runtime_seconds': runtime
}
best_model.save('../models/images/fits/cnn_mobilenet.keras')
with open('../models/images/json/cnn_mobilenet_meta.json', 'w') as f:
    json.dump(meta, f, indent=2)
print('Saved.')